## Singular Value Decomposition (SVD) from Scratch

#Singular Value Decomposition (SVD) is a powerful matrix factorization technique that decomposes any matrix $A$ into three other matrices:

#$$ A = U \Sigma V^T $$

#Where:

#*   $A$: The original $m \times n$ matrix we want to decompose.
#*   $U$: An $m \times m$ orthogonal matrix whose columns are the left singular vectors of $A$. These vectors form an orthonormal basis for the column space of $A$. Geometrically, $U$ represents a **rotation** or **reflection** in the output space.
#*   $\Sigma$: An $m \times n$ rectangular diagonal matrix containing the singular values of $A$ along its diagonal, ordered from largest to smallest. The singular values ($\sigma_i$) are non-negative scalars that indicate the "strength" or "importance" of each singular vector pair. Geometrically, $\Sigma$ represents **scaling** along the new coordinate axes.
#*   $V^T$: The transpose of an $n \times n$ orthogonal matrix $V$. The columns of $V$ (rows of $V^T$) are the right singular vectors of $A$. These vectors form an orthonormal basis for the row space of $A$. Geometrically, $V^T$ represents another **rotation** or **reflection** in the input space.

### Geometric Interpretation

#SVD can be seen as decomposing any linear transformation (represented by matrix $A$) into a sequence of three simpler geometric transformations:
#1.  A rotation/reflection ($V^T$).
#2.  A scaling along the new coordinate axes ($\Sigma$).
#3.  Another rotation/reflection ($U$).

#Essentially, SVD states that any linear transformation can be thought of as stretching and rotating space.

### Computation via Eigendecomposition

#The singular values and singular vectors are intimately related to the eigenvalues and eigenvectors of $A^T A$ and $A A^T$.

#1.  **Right Singular Vectors (V) and Singular Values ($\Sigma$)**: The columns of $V$ (right singular vectors) are the eigenvectors of the symmetric matrix $A^T A$. The squares of the singular values ($\sigma_i^2$) are the eigenvalues of $A^T A$.

 #   Given $A^T A v_i = \lambda_i v_i$, then $\sigma_i = \sqrt{\lambda_i}$.

#2.  **Left Singular Vectors (U)**: The columns of $U$ (left singular vectors) are the eigenvectors of the symmetric matrix $A A^T$. The squares of the singular values ($\sigma_i^2$) are also the eigenvalues of $A A^T$.

 #   Alternatively, once $V$ and $\Sigma$ are found, the left singular vectors $u_i$ can be derived from the relationship $A v_i = \sigma_i u_i$. Thus, for non-zero singular values, $u_i = \frac{1}{\sigma_i} A v_i$.

#For a robust implementation, it's common to compute $V$ from $A^T A$, obtain singular values from its eigenvalues, and then compute $U$ using the formula $U = A V \Sigma^{-1}$ (or element-wise $u_i = A v_i / \sigma_i$), handling cases where $\sigma_i$ might be zero or very small due to numerical precision.

### Step 2: Load and Preprocess Sample Image
This cell loads a built-in image, converts it from RGB to Grayscale by averaging the color channels, and normalizes the pixel values to a $[0, 1]$ range. This results in the matrix $A$ that we will decompose.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

# Load the 'china.jpg' sample image
china = load_sample_image("china.jpg")

# Convert to grayscale: Average across the RGB channels
# Matrix A will have dimensions (height, width)
A = np.mean(china, axis=2) / 255.0

# Display the original grayscale image
plt.figure(figsize=(8, 6))
plt.imshow(A, cmap='gray')
plt.title(f"Original Grayscale Image (Matrix A: {A.shape[0]}x{A.shape[1]})")
plt.axis('off')
plt.show()

### Step 3: Implementing SVD from Scratch
This cell defines the `svd_from_scratch` function.
1. It computes $A^T A$, which is a symmetric matrix.
2. It uses `np.linalg.eigh` (optimized for Hermitian/symmetric matrices) to find eigenvalues and eigenvectors.
3. It sorts them in descending order.
4. It calculates singular values $\sigma_i = \sqrt{\lambda_i}$.
5. It derives $U$ using the identity $u_i = \frac{A v_i}{\sigma_i}$ while handling numerical stability for small $\sigma$.

In [ ]:
def svd_from_scratch(A):
    # A is (m, n)
    m, n = A.shape

    # 1. Compute A^T * A (n x n symmetric matrix)
    # Its eigenvectors are the right singular vectors V
    ATA = A.T @ A

    # 2. Eigendecomposition of A^T * A
    # eigenvalues (lambdas) and eigenvectors (V)
    lambdas, V = np.linalg.eigh(ATA)

    # 3. Sort eigenvalues and eigenvectors in descending order
    # Note: eigh returns them in ascending order
    idx = np.argsort(lambdas)[::-1]
    lambdas = lambdas[idx]
    V = V[:, idx]

    # 4. Derive singular values: sigma = sqrt(lambda)
    # We use clip to ensure no negative values due to numerical precision
    sigmas = np.sqrt(np.maximum(lambdas, 0))

    # 5. Derive Left Singular Vectors U
    # u_i = (A * v_i) / sigma_i
    # Only for non-zero singular values
    U = np.zeros((m, len(sigmas)))
    for i in range(len(sigmas)):
        if sigmas[i] > 1e-15: # Numerical threshold
            U[:, i] = (A @ V[:, i]) / sigmas[i]

    return U, sigmas, V.T

# Run the decomposition
U, sigmas, Vt = svd_from_scratch(A)
print(f"U shape: {U.shape}, Sigma count: {len(sigmas)}, Vt shape: {Vt.shape}")

### Step 5: Image Compression using Top-k Singular Values
In this cell, we approximate the original image $A$ using only the first $k$ singular values and vectors. The rank-$k$ approximation is calculated as:

$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$

We also calculate the compression ratio, which is the ratio of the total elements in $A$ to the elements stored in $U_k, \Sigma_k, \text{ and } V_k^T$.

In [ ]:
def compress_image(U, sigmas, Vt, k):
    # Reconstruct A using only the first k singular values
    # A_k = U[:, :k] @ diag(sigmas[:k]) @ Vt[:k, :]
    U_k = U[:, :k]
    S_k = np.diag(sigmas[:k])
    Vt_k = Vt[:k, :]

    reconstructed = U_k @ S_k @ Vt_k
    return reconstructed

ks = [5, 20, 50, 100]
plt.figure(figsize=(20, 10))

# Original image size = m * n
m, n = A.shape
original_size = m * n

for i, k in enumerate(ks):
    A_k = compress_image(U, sigmas, Vt, k)

    # Compression Ratio calculation:
    # Stored values: U (m*k), sigma (k), Vt (k*n)
    compressed_size = (m * k) + k + (k * n)
    ratio = original_size / compressed_size

    plt.subplot(1, len(ks), i + 1)
    plt.imshow(A_k, cmap='gray')
    plt.title(f"k = {k}\nRatio: {ratio:.2f}:1")
    plt.axis('off')

plt.tight_layout()
plt.show()